# Exact polarization-kernel enumeration

This notebook checks the first computational layer of the GKP relative-systole project. Given an integral alternating matrix $A$, it recovers the polarization type and enumerates

$$K(L) \cong \Lambda^\perp/\Lambda = A^{-T}\mathbb Z^{2g}/\mathbb Z^{2g}$$

using exact rational arithmetic.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == 'notebooks' else cwd
src = repo_root / 'src'
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from gkp_systole import (
    KernelGroup,
    Polarization,
    canonical_alternating,
    initial_benchmarks,
    reference_uniform_relative_systole_squared,
)

print(f'Repository: {repo_root}')

Repository: .


## One-mode GKP qubit

For type $(2)$, the polarization matrix is $2J$. The kernel should contain four classes, represented by the four half-lattice points.

In [2]:
polarization = Polarization(canonical_alternating((2,)))
kernel = KernelGroup.from_polarization(polarization)

print('A =')
for row in polarization.matrix:
    print('   ', row)
print(f'type = {polarization.type}')
print(f'|K| = {kernel.order}')
print(f'exponent = {kernel.exponent}')
print('classes:')
for element in kernel.elements:
    print('   ', element.as_strings(), 'order', element.order)

assert polarization.type == (2,)
assert kernel.order == 4
assert kernel.exponent == 2

A =
    (0, 2)
    (-2, 0)
type = (2,)
|K| = 4
exponent = 2
classes:
    ('0', '0') order 1
    ('0', '1/2') order 2
    ('1/2', '0') order 2
    ('1/2', '1/2') order 2


## Current benchmark polarization types

The following checks include the types associated with the square and hexagonal one-mode codes, the $D_4$ two-mode benchmark, and the three-mode Klein-quartic benchmark. At this stage the latter two checks concern the exact finite kernel; their geometric metrics enter the next closest-vector phase.

In [3]:
rows = []
for benchmark in initial_benchmarks:
    polarization = Polarization(benchmark.alternating)
    kernel = KernelGroup.from_polarization(polarization)
    rows.append((
        benchmark.name,
        polarization.type,
        polarization.smith_factors,
        kernel.order,
        kernel.exponent,
    ))

header = ('benchmark', 'type', 'Smith factors', '|K|', 'exponent')
print(f'{header[0]:36} {header[1]:12} {header[2]:24} {header[3]:>5} {header[4]:>9}')
print('-' * 92)
for name, ptype, factors, order, exponent in rows:
    print(f'{name:36} {str(ptype):12} {str(factors):24} {order:5d} {exponent:9d}')

assert [row[3] for row in rows] == [4, 4, 4, 16, 64]

benchmark                            type         Smith factors              |K|  exponent
--------------------------------------------------------------------------------------------
square_one_mode_qubit                (2,)         (2, 2)                       4         2
hexagonal_one_mode_qubit             (2,)         (2, 2)                       4         2
one_qubit_two_modes                  (1, 2)       (1, 1, 2, 2)                 4         2
d4_type_two_mode_qubits              (2, 2)       (2, 2, 2, 2)                16         2
klein_type_three_mode_qubits         (2, 2, 2)    (2, 2, 2, 2, 2, 2)          64         2


## Square and hexagonal reference values

For uniform type $(2)$, $\Lambda^\perp=\frac12\Lambda$. A transparent bounded enumeration therefore gives a first independent check of the known values before the general closest-vector solver is added.

In [4]:
square, hexagonal = initial_benchmarks[:2]

square_ell2 = reference_uniform_relative_systole_squared(square.metric, d=2)
hexagonal_ell2 = reference_uniform_relative_systole_squared(hexagonal.metric, d=2)

print(f'square:    ell^2 = {square_ell2:.12f}, ell = {square_ell2**0.5:.12f}')
print(f'hexagonal: ell^2 = {hexagonal_ell2:.12f}, ell = {hexagonal_ell2**0.5:.12f}')
print(f'hexagonal/square ell ratio = {(hexagonal_ell2/square_ell2)**0.5:.12f}')

assert abs(square_ell2 - 0.25) < 1e-12
assert abs(hexagonal_ell2 - 1/(2 * 3**0.5)) < 1e-12
assert hexagonal_ell2 > square_ell2

square:    ell^2 = 0.250000000000, ell = 0.500000000000
hexagonal: ell^2 = 0.288675134595, ell = 0.537284965912
hexagonal/square ell ratio = 1.074569931824


## Result

The exact kernel enumeration reproduces the expected group orders and exponents. The simple uniform-type search also reproduces the square and hexagonal relative systoles. The next notebook will introduce the general closest-vector calculation for arbitrary $(A,G)$.